In [353]:
import pandas as pd
import numpy as np
import re

In [354]:
#pip install xlrd

In [355]:
#Download the file
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [356]:
#Eliminar columnas por nombre 
columns_to_drop = { 
    'pdf',
    'href',
    'href formula',
    'Case Number',
    'Case Number.1',
    'original order',
    'Unnamed: 21',
    'Unnamed: 22'
}
df = df.drop(columns=columns_to_drop)

In [357]:
df.shape

(7065, 15)

In [358]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...


#### ver los valores nulos y duplicados

In [359]:
df.isna().sum()/df.shape[0]

Date         0.000000
Year         0.000283
Type         0.002548
Country      0.007077
State        0.068931
Location     0.080255
Activity     0.082803
Name         0.030998
Sex          0.081953
Age          0.423921
Injury       0.004954
Fatal Y/N    0.079406
Time         0.499222
Species      0.443171
Source       0.002831
dtype: float64

In [360]:
df.duplicated().sum()

1

In [361]:
filas_duplicadas = df[df.duplicated(keep=False)]
filas_duplicadas

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
5436,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman
5437,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman


In [362]:
#eliminando duplicados 
df = df.drop(5436)

In [363]:
df.shape

(7064, 15)

### Column Type

In [303]:
#Looks Type 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [173]:
#Clean type
df['Type'] = df['Type'].astype(str).str.strip().str.capitalize()

#Unificar valores raros a 'Unkown
df['Type'] = df['Type'].replace({
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown'
})

#unificar 'Unprovoked' y 'Provoked' (ya los capitalizamos)
df['Type'] = df['Type'].replace({'Unprovoked': 'Unprovoked', 'Provoked': 'Provoked'})

#otros tipos 
df['Type'] = df['Type'].replace({'Watercraft': 'Other', 'Sea disaster': 'Other', 'Boat': 'Other'})

In [174]:
#Los cambios se hicieron efectivos 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other'], dtype=object)

### Hipotesis 2: Mueren mas hombres que mujeres?

In [175]:
df['Sex'].unique()

array(['M', 'F', 'F ', 'M ', nan, ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

#### Eliminacion de Nulos de ['Sex']

In [176]:
#Revisando los nulos de la columna 'Sex'
df['Sex'].isnull().sum()

579

In [177]:
#Quitamos espacios en blanco y ponemos todos en mayuscula, para unificar valores
df['Sex'] = df['Sex'].str.strip().str.upper()
df['Sex'].unique()


array(['M', 'F', nan, 'LLI', 'M X 2', 'N', '.'], dtype=object)

In [80]:
#Englobamos valores raros a 'Unkown'
df['Sex'] = df['Sex'].replace({
    'M X 2': 'Unknown',
    'LLI': 'Unknown',
    'N': 'Unknown',
    '.': 'Unknown'       
})

df['Sex'].unique()

array(['M', 'F', nan, 'Unknown'], dtype=object)

In [81]:
df['Sex'].isna().sum()

579

In [82]:
#Eliminacion de 'nan' de nuestra columna Sex
df = df.dropna(subset=['Sex'])

In [83]:
#Verificamos que se eliminaron los 'nan'
df['Sex'].unique()

array(['M', 'F', 'Unknown'], dtype=object)

In [84]:
#Utilizamos describe para ver cuando es el valor mas frecuente en Sex que es hombre
df['Sex'].describe()

count     6485
unique       3
top          M
freq      5670
Name: Sex, dtype: object

#### Hipotesis 2: En que año hay mas ataques  

In [85]:
df['Year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [86]:
#Haciendo analisis de los datos de Year y quedandonos con los años mayor a 1000
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df.loc[df['Year'] <= 1000, 'Year'] = np.nan

print("NaN:", df['Year'].isna().sum())
print("Filas antes:", df.shape[0])


NaN: 126
Filas antes: 6485


In [87]:
#Eliminando los valor "nan"
df = df.dropna(subset=['Year'])
print("Filas después:", df.shape[0])

Filas después: 6359


In [88]:
moda_year = df['Year'].mode()[0]
moda_year

2015.0

### Columna Fatal y/n

In [89]:
df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, 'Y x 2', ' N',
       'y'], dtype=object)

In [90]:
#Quitamos espacios en blanco y ponemos todos en mayuscula, para unificar valores
df['Fatal Y/N'] = df['Fatal Y/N'].str.strip().str.upper()
df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'NQ', 'UNKNOWN', 'Y X 2'], dtype=object)

In [91]:
#Englobamos valores raros a 'Unkown'
df['Fatal Y/N'] = df['Fatal Y/N'].replace({
    'Y X 2': 'Y',
    'NQ': 'N'          
})

df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'UNKNOWN'], dtype=object)

In [92]:
#Convirtiendo "UNKNOWN", "F", "M", en nan
# todo lo que NO sea Y o N pasalo a np.nan
df.loc[~df['Fatal Y/N'].isin(['Y', 'N']), 'Fatal Y/N'] = np.nan

df['Fatal Y/N'].value_counts(dropna=False)

Fatal Y/N
N      4511
Y      1323
NaN     525
Name: count, dtype: int64

In [93]:
#eliminamos los nan 
df = df.dropna(subset=['Fatal Y/N'])

In [94]:
#Visualizando cambios y utilizando describe para observar cuales son los mas frequentes 
print(df['Fatal Y/N'].unique())

df['Fatal Y/N'].describe()

['N' 'Y']


count     5834
unique       2
top          N
freq      4511
Name: Fatal Y/N, dtype: object

In [95]:
#Observando el porcentaje con respecto a la muestra final con la que nos quedamos 
(df['Fatal Y/N'].value_counts(normalize=True)['N']) * 100

77.32259170380527

### Activity

In [96]:
#Viendo los valores que contiene Activity y sus cantidades
df['Activity'].value_counts()

Activity
Surfing                                                                                                                                                                                                                                                           1073
Swimming                                                                                                                                                                                                                                                           881
Fishing                                                                                                                                                                                                                                                            371
Spearfishing                                                                                                                                                                                              

In [97]:
#revisando cuantos nulos hay en columna Activity 
df['Activity'].isna().sum()

302

In [98]:
#Los pasamos a minusculas y quitamos espacios para ver si hay coincidencias entre los valores
df['Activity'] = df['Activity'].str.strip().str.lower()

In [99]:
#ver si tenia valores nulos escritos como print y pasarlos a valor np.nan
df[df['Activity'].astype(str).str.lower().isin(['nan', 'none', 'null', ''])]['Activity'].value_counts()

Activity
    1
Name: count, dtype: int64

In [100]:
#Lista de categorías frecuentes para poder agrupar valores
frequent_categories = [
    'surfing', 'swimming', 'fishing', 'spearfishing', 'wading', 'bathing',
    'diving', 'snorkeling', 'standing', 'scuba diving', 'body boarding',
    'boogie boarding', 'body surfing', 'kayaking', 'free diving', 'treading water',
    'fell overboard', 'pearl diving', 'surf skiing', 'windsurfing', 'floating',
    'walking', 'canoeing', 'kayak fishing', 'shark fishing', 'drifting', 'boating', 'sailing'
]

In [101]:
#funcion creada para que itere en la lista de frenquent_categories y si consigue alguna palabra que este en frequent_categories
#la almacene en la categoria correspondiente, sino que la incluya en un categoria llamada other
def map_activity(activity):
    if pd.isna(activity):
        return np.nan
    for category in frequent_categories:
        if category in activity.lower():
            return category
    return 'other'


In [102]:
#aplicamos la funcion map_activy en nuestra columna y vemos el filtrado 
df['Activity'] = df['Activity'].apply(map_activity)

# Ver resultados
print(df['Activity'].value_counts(dropna=False))

Activity
surfing            1235
swimming           1107
fishing            1024
other               717
diving              475
NaN                 302
wading              178
bathing             167
standing            139
snorkeling          130
body boarding        69
boogie boarding      55
fell overboard       51
floating             46
kayaking             40
treading water       36
surf skiing          22
walking              20
canoeing             12
sailing               9
Name: count, dtype: int64


### Columna "Country"

In [103]:
#Observacion de valor unicos
df['Country'].unique()

array(['Australia', 'US Virgin Islands', 'New Caledonia', 'USA',
       'French Polynesia', 'Samoa', 'Columbia', 'Costa Rica', 'Bahamas',
       'Puerto Rico', 'Spain', 'Canary Islands', 'South Africa',
       'Vanuatu', 'Jamaica', 'Israel', 'Mexico', 'Maldives',
       'Philippines', 'Turks and Caicos', 'Mozambique', 'Egypt',
       'Thailand', 'New Zealand', 'Hawaii', 'Honduras', 'Indonesia',
       'Morocco', 'Belize', 'Maldive Islands', 'Tobago', 'AUSTRALIA',
       'INDIA', 'TRINIDAD', 'BAHAMAS', 'SOUTH AFRICA', 'MEXICO',
       'NEW ZEALAND', 'EGYPT', 'BELIZE', 'Coral Sea', 'SPAIN', 'PORTUGAL',
       'SAMOA', 'COLOMBIA', 'ECUADOR', 'FRENCH POLYNESIA',
       'NEW CALEDONIA', 'TURKS and CaICOS', 'CUBA', 'BRAZIL', 'FIJI',
       'MeXICO', 'ENGLAND', 'JAPAN', 'INDONESIA', 'JAMAICA', 'MALDIVES',
       'THAILAND', 'COLUMBIA', 'British Overseas Territory', 'CANADA',
       'JORDAN', 'ST KITTS / NEVIS', 'ST MARTIN', 'SEYCHELLES',
       'PAPUA NEW GUINEA', 'ISRAEL', 'REUNION ISLAND', 

In [104]:
#unificando para que todo sea mayuscula y quitando espacio en blanco 
df['Country'] = df['Country'].str.upper().str.strip()

In [105]:
#Observamos que se unifican algunos valores 
df['Country'].value_counts()

Country
USA                                      2267
AUSTRALIA                                1238
SOUTH AFRICA                              468
BAHAMAS                                   127
NEW ZEALAND                               120
PAPUA NEW GUINEA                          112
BRAZIL                                     98
MEXICO                                     89
FIJI                                       60
REUNION                                    52
NEW CALEDONIA                              50
ITALY                                      47
PHILIPPINES                                46
EGYPT                                      45
CUBA                                       41
MOZAMBIQUE                                 38
SPAIN                                      33
FRENCH POLYNESIA                           33
PANAMA                                     29
INDIA                                      28
JAMAICA                                    28
SOLOMON ISLANDS           

In [106]:
#Aqui toma el primer valor al separa valor/valor
df['Country'] = df['Country'].str.split('/').str[0].str.strip()

In [107]:
#lo pasamos a series para observar con cuantos valores unicos estamos tratando 
pd.Series(df['Country'].unique())

0                                  AUSTRALIA
1                          US VIRGIN ISLANDS
2                              NEW CALEDONIA
3                                        USA
4                           FRENCH POLYNESIA
5                                      SAMOA
6                                   COLUMBIA
7                                 COSTA RICA
8                                    BAHAMAS
9                                PUERTO RICO
10                                     SPAIN
11                            CANARY ISLANDS
12                              SOUTH AFRICA
13                                   VANUATU
14                                   JAMAICA
15                                    ISRAEL
16                                    MEXICO
17                                  MALDIVES
18                               PHILIPPINES
19                          TURKS AND CAICOS
20                                MOZAMBIQUE
21                                     EGYPT
22        

In [ ]:
# Países con múltiples nombres o abreviaciones
country_mapping = {    
    'ST KITTS / NEVIS': 'SAINT KITTS AND NEVIS',
    'ST KITTS': 'SAINT KITTS AND NEVIS',
    'NEVIS': 'SAINT KITTS AND NEVIS',
    'TRINIDAD & TOBAGO': 'TRINIDAD AND TOBAGO',
    'TRINIDAD': 'TRINIDAD AND TOBAGO',
    'TOBAGO': 'TRINIDAD AND TOBAGO',
    'TURKS & CAICOS': 'TURKS AND CAICOS',
    'TURKS': 'TURKS AND CAICOS',
    'ST. MARTIN': 'SAINT MARTIN',
    'ST. MAARTIN': 'SAINT MARTIN',
    'MALDIVE ISLANDS': 'MALDIVES',
    'CEYLON': 'SRI LANKA',
    'CEYLON (SRI LANKA)': 'SRI LANKA',
    'COLUMBIA': 'COLOMBIA',
    'OKINAWA': 'JAPAN',
    'WESTERN SAMOA': 'SAMOA',
    'BRITISH NEW GUINEA': 'PAPUA NEW GUINEA',
    'NEW BRITAIN': 'PAPUA NEW GUINEA',
    
    # Territorios y países especiales
    'BRITISH VIRGIN ISLANDS': 'UNITED KINGDOM',
    'ST HELENA, BRITISH OVERSEAS TERRITORY': 'UNITED KINGDOM',
    'UNITED ARAB EMIRATES (UAE)': 'UNITED ARAB EMIRATES',
    'HAWAII': 'USA',
    'CANARY ISLANDS': 'SPAIN',
    'ANDAMAN ISLANDS': 'INDIA',
    'GUAM (US TERRITORY)': 'USA',
    
    # Zonas geográficas no país → OTHER
    'CORAL SEA': 'OTHER',
    'ATLANTIC OCEAN': 'OTHER',
    'CARIBBEAN SEA': 'OTHER',
    'COAST OF AFRICA': 'OTHER',
    'RED SEA?': 'OTHER',
    'ASIA?': 'OTHER',
    'DIEGO GARCIA': 'OTHER',
    'REUNION ISLAND': 'OTHER',
    'GULF OF ADEN': 'OTHER',
    'TASMAN SEA': 'OTHER',
    'NORTH ATLANTIC OCEAN': 'OTHER',
    'SOUTH CHINA SEA': 'OTHER',
    'PACIFIC OCEAN': 'OTHER',
    'JOHNSTON ISLAND': 'OTHER',
    'SOUTH PACIFIC OCEAN': 'OTHER',
    'WEST INDIES': 'OTHER',
    'OCEAN': 'OTHER',
    'MEDITERRANEAN SEA': 'OTHER',
    'ROATAN': 'OTHER',
    'NORTHERN ARABIAN SEA': 'OTHER',
    'INDEPENDENT STATES': 'OTHER',
    'KOREA': 'OTHER'
}

In [109]:
df['Country'] = df['Country'].replace(country_mapping)

In [110]:
df['Country'].unique()

array(['AUSTRALIA', 'US VIRGIN ISLANDS', 'NEW CALEDONIA', 'USA',
       'FRENCH POLYNESIA', 'SAMOA', 'COLOMBIA', 'COSTA RICA', 'BAHAMAS',
       'PUERTO RICO', 'SPAIN', 'SOUTH AFRICA', 'VANUATU', 'JAMAICA',
       'ISRAEL', 'MEXICO', 'MALDIVES', 'PHILIPPINES', 'TURKS AND CAICOS',
       'MOZAMBIQUE', 'EGYPT', 'THAILAND', 'NEW ZEALAND', 'HONDURAS',
       'INDONESIA', 'MOROCCO', 'BELIZE', 'TRINIDAD AND TOBAGO', 'INDIA',
       'OTHER', 'PORTUGAL', 'ECUADOR', 'CUBA', 'BRAZIL', 'FIJI',
       'ENGLAND', 'JAPAN', 'BRITISH OVERSEAS TERRITORY', 'CANADA',
       'JORDAN', 'SAINT KITTS AND NEVIS', 'ST MARTIN', 'SEYCHELLES',
       'PAPUA NEW GUINEA', 'CHINA', 'IRELAND', 'ITALY', 'MALAYSIA', nan,
       'MAURITIUS', 'SOLOMON ISLANDS', 'UNITED KINGDOM', 'REUNION',
       'UNITED ARAB EMIRATES', 'DOMINICAN REPUBLIC', 'ARUBA',
       'SAINT MARTIN', 'FRANCE', 'KIRIBATI', 'TAIWAN',
       'PALESTINIAN TERRITORIES', 'GUAM', 'NIGERIA', 'TONGA', 'SCOTLAND',
       'CHILE', 'KENYA', 'RUSSIA', 'AZORES',

In [111]:
pd.Series(df['Country'].unique())

0                           AUSTRALIA
1                   US VIRGIN ISLANDS
2                       NEW CALEDONIA
3                                 USA
4                    FRENCH POLYNESIA
5                               SAMOA
6                            COLOMBIA
7                          COSTA RICA
8                             BAHAMAS
9                         PUERTO RICO
10                              SPAIN
11                       SOUTH AFRICA
12                            VANUATU
13                            JAMAICA
14                             ISRAEL
15                             MEXICO
16                           MALDIVES
17                        PHILIPPINES
18                   TURKS AND CAICOS
19                         MOZAMBIQUE
20                              EGYPT
21                           THAILAND
22                        NEW ZEALAND
23                           HONDURAS
24                          INDONESIA
25                            MOROCCO
26          

In [112]:
df['Country'].value_counts()

Country
USA                               2268
AUSTRALIA                         1238
SOUTH AFRICA                       468
BAHAMAS                            127
NEW ZEALAND                        120
PAPUA NEW GUINEA                   119
BRAZIL                              98
MEXICO                              89
FIJI                                60
REUNION                             52
NEW CALEDONIA                       50
OTHER                               49
ITALY                               47
EGYPT                               46
PHILIPPINES                         46
CUBA                                41
MOZAMBIQUE                          38
SPAIN                               34
FRENCH POLYNESIA                    33
JAPAN                               31
PANAMA                              29
INDIA                               29
JAMAICA                             28
SOLOMON ISLANDS                     27
IRAN                                26
HONG KONG        

### Columna Date

In [364]:
df['Date'].unique()


array(['10th January', '8th January', '3rd January ', ..., '1900-1905',
       '1883-1889', '1845-1853'], dtype=object)

In [365]:
def false_words(s):
    eliminate_words = [
        'Before', 'After', 'No date', 'Late',
        'Circa', 'Ca.', 'Letter dated'
    ]
    
    s = str(s).lower()
    #Recorre la lista eliminate_words comprueba si alguna de esas palabras esta dentro del texto 
    #devuelve True si al menos encuenta una 
    for word in eliminate_words:
        if word.lower() in s:
            return True
    
    return False

In [366]:
def extract_full_month(s):
    #lista de los meses
    months_full = [
        'January', 'February', 'March', 'April', 'May', 'June',
        'July', 'August', 'September', 'October', 'November', 'December'
    ]
    
    #
    months_abbreviate = []
    for m in months_full:
        months_abbreviate.append(m[:3])
    #flags=re.I -> hace la bisque a mayuscula y minusculas
    #crea pares con sus el nombre completo del mes y su abreviatura ('January', 'Jan')
    for m_full, m_abbr in zip(months_full, months_abbreviate):
        #primera condicion busca el nombre completo del mes y la segunda la abreviatura
        if re.search(rf'\b{m_full}\b', s, flags=re.I) or \
           re.search(rf'\b{m_abbr}\b', s, flags=re.I):
            return m_full
    
    return None

In [367]:
#esta funcion trabaja con un solo valor
#su trabajo es limpiar, validar y extraer el mes 
def parse_month(x):
    if pd.isna(x):
        return None
        
    s = str(x).strip()
        
    #Descartar fechas inciertas
    if false_words(s):
        return None
        
    #Extraer mes
    return extract_full_month(s)

In [368]:
def extract_full_month_strict(date_series):
    return date_series.apply(parse_month)

In [369]:
df['Month_full_strict'] = extract_full_month_strict(df['Date'])

In [370]:
print(df[['Date', 'Month_full_strict']])

                                                   Date Month_full_strict
0                                          10th January           January
1                                           8th January           January
2                                          3rd January            January
3                                         21st December          December
4                                         12th December          December
5                                          9th December          December
6                                        27th November           November
7                                        27th November           November
8                                         10th November          November
9                                          9th November          November
10                                         5th November          November
11                                         5th November          November
12                                    

In [371]:
#con dropna=False podemos ver que tenemos None y hay que eliminarlos
df['Month_full_strict'].value_counts(dropna=False)

Month_full_strict
July         775
August       667
None         637
September    608
January      568
June         547
October      502
December     493
April        491
November     457
March        456
May          447
February     416
Name: count, dtype: int64

In [372]:
#.notna() es un metodo que sirve para detectar los valores que no son nulos 
#el valor existe devuelve True, sino False
df = df[df['Month_full_strict'].notna()]

In [373]:
#Revisando que eliminamos los None
df['Month_full_strict'].value_counts(dropna=False)

Month_full_strict
July         775
August       667
September    608
January      568
June         547
October      502
December     493
April        491
November     457
March        456
May          447
February     416
Name: count, dtype: int64